<a href="https://colab.research.google.com/github/M9249/Function.py/blob/main/app_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import streamlit as st
import cv2
import numpy as np
from PIL import Image
import torch
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn

# Page Config
st.set_page_config(page_title="Smart Agrivision: Crop Disease & Severity Analyzer", layout="wide")

# App Header
st.title("🌱 Smart Agriculture: Crop Leaf Disease & Severity Analyzer")
st.write("Upload a leaf image to automatically classify the disease and compute its infection severity percentage.")

# -------------------------------------------------------------
# 1. Load Model & Transformations (Cached for performance)
# -------------------------------------------------------------
@st.cache_resource
def load_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Define class labels matching your dataset folder structure
    class_labels = [
        "Apple Scab",
        "Black Rot",
        "Cedar Apple Rust",
        "Healthy Leaf",
        "Powdery Mildew"
    ] # Update these to match your actual dataset folder names!

    # Initialize ResNet-18 structure
    model = models.resnet18(weights=None)
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_ftrs, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, len(class_labels))
    )

    # Optional: Load your trained weights if saved (e.g., model.load_state_dict(torch.load('model_weights.pth')))
    model.eval()
    return model, class_labels, device

model, class_labels, device = load_model()

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# -------------------------------------------------------------
# 2. Severity Estimation Function (HSV Pipeline)
# -------------------------------------------------------------
def estimate_disease_severity(image_np):
    hsv = cv2.cvtColor(image_np, cv2.COLOR_RGB2HSV)

    # Isolate leaf foliage
    lower_leaf = np.array([15, 20, 20])
    upper_leaf = np.array([85, 255, 255])
    leaf_mask = cv2.inRange(hsv, lower_leaf, upper_leaf)

    # Isolate diseased spots
    lower_disease = np.array([5, 40, 20])
    upper_disease = np.array([28, 255, 220])
    disease_mask = cv2.inRange(hsv, lower_disease, upper_disease)

    disease_mask = cv2.bitwise_and(disease_mask, disease_mask, mask=leaf_mask)

    total_leaf_pixels = cv2.countNonZero(leaf_mask)
    diseased_pixels = cv2.countNonZero(disease_mask)

    if total_leaf_pixels == 0:
        return 0.0, "No leaf detected", disease_mask

    severity_pct = (diseased_pixels / total_leaf_pixels) * 100

    if severity_pct < 3.0:
        stage = "Healthy / Negligible"
    elif severity_pct < 15.0:
        stage = "Mild Infection (Stage 1)"
    elif severity_pct < 35.0:
        stage = "Moderate Infection (Stage 2)"
    else:
        stage = "Severe Infection (Stage 3)"

    return round(severity_pct, 2), stage, disease_mask

# -------------------------------------------------------------
# 3. UI Layout & Image Upload Handler
# -------------------------------------------------------------
uploaded_file = st.file_uploader("Choose a crop leaf image...", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    # Read Image
    image = Image.open(uploaded_file).convert('RGB')
    image_np = np.array(image)

    col1, col2 = st.columns(2)

    with col1:
        st.subheader("Uploaded Image")
        st.image(image, use_column_width=True)

    with st.spinner("Analyzing leaf health and severity..."):
        # Run Classification
        input_tensor = val_transform(image).unsqueeze(0).to(device)
        with torch.no_grad():
            outputs = model(input_tensor)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)
            confidence, predicted_idx = torch.max(probabilities, 1)

        pred_class = class_labels[predicted_idx.item()]
        conf_score = round(confidence.item() * 100, 2)

        # Run Severity Analysis
        severity_pct, stage, disease_mask = estimate_disease_severity(image_np)

    with col2:
        st.subheader("Diagnostic Results")
        st.metric(label="Predicted Condition", value=pred_class, delta=f"{conf_score}% Confidence")
        st.metric(label="Infection Severity Index", value=f"{severity_pct}%", delta=stage, delta_value_color="inverse")

        st.markdown("### **Actionable Recommendation:**")
        if "Healthy" in stage or severity_pct < 3.0:
            st.success("The plant appears healthy. Maintain standard monitoring practices.")
        elif "Mild" in stage:
            st.warning("Early stage infection detected. Consider targeted organic treatment or localized removal of affected leaves.")
        else:
            st.error("Moderate-to-severe infection detected. Immediate fungicide application or agricultural consultation recommended.")

    # Show Lesion Mask Visualization
    st.markdown("---")
    st.subheader("Advanced Vision Insights: Lesion Isolation Mask")
    st.image(disease_mask, caption="Heatmap showing quantified diseased pixel regions", use_column_width=False, width=400)

ModuleNotFoundError: No module named 'streamlit'